<a href="https://colab.research.google.com/github/khine-thant-su/crisis_companion_chatbot/blob/main/safety_classifier_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Description of notebook contents
This notebook contains code to train a classifier model that classifies a given user input into one of four categories - "depression", "SuicideWatch", "teenagers" and "other".

Dataset used for training the classifier: [Suicide Depression Detection](https://huggingface.co/datasets/joshyii/suicide_depression_detection)

This safety classifier can be integrated with the chatbot as follows:

* User input -> Classifier -> Risk scores (Predicted probabilities of a response belonging to a class) -> Apply thresholds for follow-up action.


In [4]:
import pandas as pd
import pyarrow.parquet as pq  # To read the parquet file

import string  # To preprocess text data
import nltk
from nltk.stem import WordNetLemmatizer as wnl
from nltk.corpus import wordnet
from nltk.tokenize import word_tokenize

# Download necessary NLTK data
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger_eng')  # For part-of-speech (POS) tagging
nltk.download('punkt_tab')  # For sentence tokenization

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [5]:
RANDOM_STATE = 42

In [6]:
# Read the Parquet file into an Arrow Table
table = pq.read_table('/content/0000.parquet')

# Convert the Arrow Table to a Pandas DataFrame
df = table.to_pandas()
print(df)

                                                     text         class
0       Does life actually work for most / non-depress...    depression
1       I found my friend's bodyIt was almost nine yea...    depression
2       Ex Wife Threatening SuicideRecently I left my ...  SuicideWatch
3       Am I weird I don't get affected by compliments...     teenagers
4       Finally 2020 is almost over... So I can never ...     teenagers
...                                                   ...           ...
348119  You how you can tell i have so many friends an...     teenagers
348120  pee probably tastes like salty tea😏💦‼️ can som...     teenagers
348121  The usual stuff you find hereI'm not posting t...  SuicideWatch
348122  I confronted my mother. Extremely isolated, wi...    depression
348123  I still haven't beaten the first boss in Hollo...     teenagers

[348124 rows x 2 columns]


In [7]:
# Check for missing values in either column
df.isna().sum()

,0
text,1
class,14


In [8]:
# Observations with missing class -- These could be used as test data maybe?
df.loc[df['class'].isna()]

,text,class
11557,I feel like im in a nightmare.Something happen...,None
11558,It's like I'm living in a nightmare and everyt...,None
11559,(view post history for more info on my dad),None
41048,A doodle of my struggle with depressionhttp://...,None
47570,Thinking of putting this as my profile picture...,None
61160,If I told you I want to move on with my life a...,None
141715,I think I might need someone to talk me down f...,None
141716,I've known that I'll never get any love outsid...,None
156657,A clip that describes how I feel when I'm tryi...,None
156658,depression,None


In [9]:
# Observation with missing text
df.loc[df['text'].isna()]

,text,class
185323,None,depression


In [10]:
# Check for very short texts that might be noise
print("Number of short text entries:", len(df.loc[df['text'].str.strip().str.len() < 10]), "\n")
df.loc[df['text'].str.strip().str.len() < 10].head()

Number of short text entries: 35 



,text,class
11019,okok,SuicideWatch
19013,f you :),teenagers
24908,Hello:],SuicideWatch
28983,Hello.:),SuicideWatch
29193,deadme?,SuicideWatch


In [11]:
# There is equal class distribution across the three classes.
df['class'].value_counts()

,count
class,
SuicideWatch,116037
teenagers,116037
depression,116036


In [12]:
# Check for duplicate entries in the 'text' column -- no duplicate prompts.
duplicate_entries = df[df['text'].duplicated(keep=False)]
display(duplicate_entries)

,text,class


In [13]:
df.sample(3, random_state=RANDOM_STATE)['text']

,text
182709,I never thought I would be cheated on. But her...
18177,should i switch from eclipse to visual studio ...
252758,Good in this evil worldI've been through pain ...


In [14]:
# Prepare subset dataset before train_test_split
df_subset = df[(df['class'].notna()) & (df['text'].notna())]  # Drop rows where class or text is missing
df_subset = df_subset[df_subset['text'].str.strip().str.len() > 10]  # Drop rows where text is too short
len(df_subset)

348061

In [15]:
df_subset['class'].value_counts()

,count
class,
teenagers,116028
depression,116026
SuicideWatch,116007


### Train-val-test split

In [16]:
# Train-val-test split
X = df_subset['text'].values
y = df_subset['class'].values

# 80% temp (train + val), 20% test
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)  # Ensures that the split is performed in a way that maintains the same proportion of classes in both the training and testing datasets as in the original dataset

# Split the 80% into 70% train, 10% val (10% of the original dataset will be saved for validation)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.125, random_state=RANDOM_STATE, stratify=y_temp)  # 0.8 * 0.125 = 0.10


In [17]:
print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

Train: 243642, Val: 34806, Test: 69613


In [18]:
# Preview the first few elements of X_train
# X_train is an array of strings.
display(X_train[120])

'Question about tooth extraction Does it hurt too much when the dentist removes a tooth that is not loose? Seriously, I do not know where to ask this, but I need a positive answer to relieve my stress.'

In [19]:
# Create subset arrays for training because otherwise RAM will run out.
X_train_subset = X_train[:5000]
y_train_subset = y_train[:5000]

Build pipeline (TF-IDF → Logistic Regression)

In [20]:
# Example for how nltk.pos_tag() works
# print(nltk.pos_tag(['feet']))
# print(nltk.pos_tag(['feet'])[0][1][0])

In [21]:
from nltk.corpus import wordnet
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords # Import stopwords
import nltk
from nltk.stem import WordNetLemmatizer as wnl
import string

# Initialize WordNetLemmatizer
lemmatizer = wnl()

def get_wordnet_pos(word):
    """Map NLTK POS tags to WordNet POS tags"""
    tag = nltk.pos_tag([word])[0][1][0].upper()  # Extract the first letter
    tag_dict = {"J": wordnet.ADJ,
                "N": wordnet.NOUN,
                "V": wordnet.VERB,
                "R": wordnet.ADV}
    return tag_dict.get(tag, wordnet.NOUN) # Default to noun if POS not found

# Modify the clean function to include POS tagging and stopword removal
def clean(text):
    '''Converts the input text into lower case, removes <br> tags, punctuation, whitespace, and stopwords. Lemmatizes the words using POS tags.
    Returns the processed words in a list.

        Args:
        text(str): input text'''

    text = "".join([i.lower() for i in text if i not in string.punctuation])  # Only keep non-punctuation characters. Each "i" is a letter, not a word. "".join() returns a sentence.
    words = word_tokenize(text)  # Tokenize the text into words
    words = [word for word in words if word not in stopwords.words('english')]  # Remove stopwords
    text = ' '.join([lemmatizer.lemmatize(word, get_wordnet_pos(word)) for word in words])  # Lemmatize words based on their POS tags

    return text

### Preprocess text before feeding it into TfidfVectorizer

In [23]:
X_train_subset_cleaned = [clean(text) for text in X_train_subset]
display(X_train_subset_cleaned[120])

'question tooth extraction hurt much dentist remove tooth loose seriously know ask need positive answer relieve stress'

In [24]:
# 1. Initialize TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer()

# 2. Fit and transform the training data
# fit() learns the vocabulary and IDF values
# transform() converts the text to TF-IDF features
tfidf_matrix = tfidf_vectorizer.fit_transform(X_train_subset_cleaned)  # tfidf_matrix will be a sparse matrix

# Convert to a dense array for easier viewing
print(tfidf_matrix.toarray()[:10])

# Get the feature names. A feature is a unique word or a sequence of words (n-grams) in the vocabulary that TfidfVectorizer has built, based on the documents it's been fitted on.
feature_names = tfidf_vectorizer.get_feature_names_out()
print("\nFeature Names (Vocabulary):\n")
print(feature_names[:100])

[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]

Feature Names (Vocabulary):

['00' '000' '001g' ... 'ಠಠ' '私は同性愛者です' '𓁹𓂏𓁹𓂈']


In [26]:
feature_names[:20]

array(['00', '000', '001g', '003478628', '0125', '02', '0240', '0300',
       '032015', '04', '0414', '0421', '04210', '05mg', '06', '07', '10',
       '100', '1000', '10000'], dtype=object)

In [29]:
# Check why there are numbers in the text data
for text in X_train_subset_cleaned:
  if '003478628' in text:
    display(text)

'adios amigo 003478628 hallelujah world great journey first bangladesh english good youre see mean im deadi longer tangent universe know dont time read also celebritybut year ive gather huge amount knowledgeand last day dimension something confessfirst like personsi lot differenti think different tooor wont able heretheyre try terminate mebut know ill always herei cant write book life wont appreciate appreciationtoday human much foolishthey whatever theyre told talk whatever theyre taughti philosophisti nobodyi wish collective mindthis last day cant type everythingjust dont theyve implant best note youll ever readsomethings go like big conspiracykeep mind open eye earsi hope could exist little bit long cantwhether exist doesnt matter anybodythey want perishweve make solve itlove mom one love mostmy little sister know shell harvard someday certainly willi dont want recall bad thing life last want say adios amigo'

Should I get rid of the numbers in the text?